In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Auto Alignment Evaluation of LLM Output

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/jenniferliangc/Auto-alignment-evaluation-of-LLM-output/blob/main/auto_alignment_evaluation_of_llm_output.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fjenniferliangc%2FAuto-alignment-evaluation-of-LLM-output%2Fmain%2F2025%2Fgenerative-ai%2Fgemini%2Fevaluation%2Fauto_alignment_evaluation_of_llm_output.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/jenniferliangc/Auto-alignment-evaluation-of-LLM-output/blob/main/auto_alignment_evaluation_of_llm_output.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/jenniferliangc/Auto-alignment-evaluation-of-LLM-output/blob/main/auto_alignment_evaluation_of_llm_output.ipynb">
      <img width="32px" src="https://www.svgrepo.com/download/217753/github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/jenniferliangc/Auto-alignment-evaluation-of-LLM-output/blob/main/auto_alignment_evaluation_of_llm_output.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/jenniferliangc/Auto-alignment-evaluation-of-LLM-output/blob/main/auto_alignment_evaluation_of_llm_output.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/jenniferliangc/Auto-alignment-evaluation-of-LLM-output/blob/main/auto_alignment_evaluation_of_llm_output.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/jenniferliangc/Auto-alignment-evaluation-of-LLM-output/blob/main/auto_alignment_evaluation_of_llm_output.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/jenniferliangc/Auto-alignment-evaluation-of-LLM-output/blob/main/auto_alignment_evaluation_of_llm_output.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>

| Author(s) |
| --- |
| [Jennifer Liang](https://github.com/jenniferliangc) |

## Overview

This tutorial brings a new way to evaluate LLM performance against ground truth. We build a customizable, line-by-line automated evaluator for use cases where high precision is required. This method eliminates the need for repetitive prompt tuning, minimizes hallucinations, and ensures repeatable and accurate results.


In this tutorial, we use a recipe dataset that has ground truth and LLM outputs. We will perform two phases of evaluation:

1. Rephraser Evaluator (0-1 point): use a semantic similarity model to check similarity on LLM output vs ground truth.

2. Final Answer Evaluator (0-5 points): we have three criterias to evaluate in this phase:
> - a) Source Score (0-1 point): did the LLM choose the same source (source 1, 2, 3, etc.) as in the ground truth?
> - b) Ingredient Sentence Score (0-2 points): sentence-level comparison to check if LLM outputted the same ingredient list as in the ground truth.
> - c) Instruction Sentence Score (0-2 points): sentence-level comparison to check if LLM outputted the same instructions as in the ground truth.
> - For a, b, and c, penalties are added for any missed ground truth sources or sentences. A penalty is also added for any extra sources or sentences that the LLM produces not present in the ground truth

## Get started

### Install Google Gen AI SDK and other required packages


In [ ]:
# %pip install --upgrade --quiet google-cloud-aiplatform

In [ ]:
# %pip install google-genai --quiet

In [ ]:
# %pip install tf-keras --quiet

In [ ]:
# %pip install sentence-transformers --quiet

In [ ]:
import vertexai
from vertexai.language_models import TextEmbeddingModel

### Authenticate your notebook environment (Colab only)

If you're running this notebook on Google Colab, run the cell below to authenticate your environment.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Set Google Cloud project information

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [ ]:
# Use the environment variable if the user doesn't provide Project ID.
import os

PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "us-central1")

from google import genai

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

### Import libraries

In [ ]:
vertexai.init(project=PROJECT_ID, location=LOCATION)

In [ ]:
from IPython.display import Markdown, display

import json
import logging
import time
import uuid
import pandas as pd
import numpy as np
import os, sys, vertexai
import regex as re
import json
import ast
import requests
from typing import NamedTuple
from google.cloud import aiplatform

# Logging
logger = logging.getLogger("logger")
logging.basicConfig(level=logging.WARNING)

## Custom Rater Evaluator

### Load Dataset

In [ ]:
sep = ','

In [ ]:
df = pd.read_csv("dataset.csv", sep=sep)
print(df.shape)
df.head(2)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util

In [ ]:
import vertexai
from vertexai.language_models import TextEmbeddingModel
MODEL_ID = 'text-embedding-004'
model = TextEmbeddingModel.from_pretrained(MODEL_ID)

### Rephraser Evaluator

In [ ]:
gt_rephrased_query =  'ground_truth_rephrased_query'
llm_rephrased_query = 'gemini_alternative_rephrased_query_llm'

In [ ]:
import vertexai
from vertexai.language_models import TextEmbeddingModel
from sentence_transformers import util

def calculate_similarity(model, llm_rephraser, gt_rephraser):
    """Calculates the cosine similarity between two text embeddings.

    Args:
        llm_rephraser: The first text string.
        gt_rephraser: The second text string.

    Returns:
        The cosine similarity between the embeddings of the two text strings.
    """
    embeddings1 = model.get_embeddings([llm_rephraser])
    embeddings2 = model.get_embeddings([gt_rephraser])
    cosine_sim = util.cos_sim(embeddings1[0].values, embeddings2[0].values)
    return cosine_sim.item()

In [ ]:
df['rephraser_semantic_similarity'] = df.apply(lambda row: calculate_similarity(model, row[llm_rephrased_query], row[gt_rephrased_query]), axis=1)

In [ ]:
df.head(2)

### Final Answer Evaluator

In [ ]:
gt_final_answer = 'ground_truth_final_answer'
llm_final_answer = 'gemini_alternative_answer'

#### Source Scoring

In [ ]:
# Extract Recipe Number Source

def find_recipe_number(text):
    """
    Finds and extracts unique recipe numbers from a given text string.

    The function searches for the pattern "Recipe " followed by one or more digits.
    It returns a sorted list of the unique recipe numbers found in the text.
    If the input text is NaN or if no recipe numbers are found, an empty list is returned.

    Args:
        text (str): The input string to search for recipe numbers.

    Returns:
        list: A sorted list of unique recipe numbers (as strings) found in the text.
              Returns an empty list if no recipe numbers are found or if the input
              text is pandas.NA.
    """
    if pd.isna(text):
        return []
    else:
        recipe_numbers = []
        pattern = r"Recipe (\d+)"
        matches = re.findall(pattern, text)
        if matches:
            recipe_numbers.extend(matches)
            return sorted(list(set(recipe_numbers)))
        else:
            return []

In [ ]:
df['gt_sources_extracted'] = df[gt_final_answer].apply(find_recipe_number)
df['llm_response_sources'] = df[llm_final_answer].apply(find_recipe_number)

In [ ]:
df.head(2)

In [ ]:
# SOURCE SCORE CALCULATION

def source_score(llm_sources, gt_sources):
    """
    Calculates a source score by comparing the sources provided by an LLM
    to the ground truth (GT) sources.

    The score is based on the number of correctly identified sources, with penalties
    for incorrect LLM-provided sources and missed ground truth sources.

    Args:
        llm_sources (list): A list of sources provided by the LLM.
        gt_sources (list): A list of ground truth sources.

    Returns:
        tuple: A tuple containing the following five float values:
            - source_score: The calculated source score (capped at 0).
            - points_per_source: The base points awarded for each correct ground truth source.
            - correct_sources_score: The total score from correctly identified sources
              before penalties.
            - incorrect_source_penalty: The penalty applied for incorrect sources
              provided by the LLM.
            - missed_source_penalty: The penalty applied for ground truth sources
              that were not identified by the LLM.
    """

    if not llm_sources or not gt_sources:
        return 0, 0, 0, 0, 0

    # SOURCE SCORE CALCULATION

    # Calculate points per source
    if len(gt_sources) == 0:
        points_per_source = 0.00
    else:
        points_per_source = round(1 / len(gt_sources), 2)

    # Count correct and incorrect matches using set operation
    correct_sources = len(set(gt_sources) & set(llm_sources))
    incorrect_llm_sources = len(set(llm_sources) - set(gt_sources)) #finds the values that are in llm_sources but NOT in gt_sources
    missed_gt_sources = len(set(gt_sources) - set(llm_sources)) #finds the values that are in gt_sources but NOT in llm_sources

    # Define penalty values (adjust penalty % as needed)
    incorrect_source_penalty = 0.10 * points_per_source * incorrect_llm_sources # Per incorrect LLM source
    missed_source_penalty = 0.20 * points_per_source * missed_gt_sources # Per missed GT source
    correct_sources_score = correct_sources * points_per_source

    # Calculate the score and ensure it doesn't go below zero
    source_score = round(max(0, (correct_sources_score -
                                 incorrect_source_penalty -
                                 missed_source_penalty)), 2)

    return source_score, points_per_source, correct_sources_score, incorrect_source_penalty, missed_source_penalty

In [ ]:
df[['source_score', 'points_per_source', 'correct_sources_score', 'incorrect_source_penalty', 'missed_source_penalty']] = df.apply(
lambda row: source_score(row['llm_response_sources'], row['gt_sources_extracted']), axis=1, result_type="expand")

#### Ingredients and Instruction Scoring

In [ ]:
def extract_instructions_regex(text: str) -> str:
    # Regex to find text after "Instructions:"
    match = re.search(r"instructions?\s*(.*)", text, re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).strip().replace("{", "").replace("}", "").replace("•", "").replace(":", "")
    return ""

def extract_ingredients_regex(text):
    # Regex to find text between "Ingredients" and "Instructions"
    match = re.search(r"ingredients?\s*(.*?)(?:\s*[\"'\n]*\s*instructions:?)", text, re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).strip().replace("{", "").replace("}", "").replace("•", "").replace(":", "")
    return ""

def extract_ingredients_and_instructions(llm_text, gt_text, llm_sources, gt_sources):
    """
    Extracts ingredients and instructions from LLM-generated text and ground truth text,
    focusing on the content associated with correctly identified sources.

    The function first identifies the common sources between the LLM's output and the
    ground truth. Then, it extracts the full ingredients and instructions blocks from
    both the LLM text and the ground truth text using regular expressions
    (via `extract_ingredients_regex` and `extract_instructions_regex`).
    The extracted ingredient and instruction blocks are then duplicated for each
    correctly identified source, creating parallel lists.

    Args:
        llm_text (str): The text generated by the Language Model.
        gt_text (str): The ground truth text.
        llm_sources (List[str]): A list of sources cited by the LLM.
        gt_sources (List[str]): A list of ground truth sources.

    Returns:
        Tuple[List[str], List[str], List[str], List[str]]: A tuple containing four lists:
            - llm_ingredients_text (List[str])
            - llm_instructions_text (List[str])
            - gt_ingredients_text (List[str])
            - gt_instructions_text (List[str])
            Returns four empty lists if any of the input texts or source lists are empty.
    """

    if not llm_text or not llm_sources or not gt_sources:
        return [], [], [], []

    correct_sources = sorted(list(set(gt_sources) & set(llm_sources)))

    llm_ingredients_text = []
    gt_ingredients_text = []

    llm_instructions_text = []
    gt_instructions_text = []

    # --- Process LLM Text ---
    llm_ingredients_block = extract_ingredients_regex(llm_text)
    if not llm_ingredients_block:
        llm_ingredients_block = ""
    llm_instructions_block = extract_instructions_regex(llm_text)
    if not llm_instructions_block:
        llm_instructions_text = ""

    # --- Process GT Text ---
    gt_ingredients_block = extract_ingredients_regex(gt_text)
    if not gt_ingredients_block:
        gt_ingredients_block = ""
    gt_instructions_block = extract_instructions_regex(gt_text)
    if not gt_instructions_block:
        gt_instructions_block = ""

    for source in correct_sources:
        llm_ingredients_text.append(llm_ingredients_block)
        llm_instructions_text.append(llm_instructions_block)

        gt_ingredients_text.append(gt_ingredients_block)
        gt_instructions_text.append(gt_instructions_block)

    return llm_ingredients_text, llm_instructions_text, gt_ingredients_text, gt_instructions_text

In [ ]:
df[['llm_ingredients_text', 'llm_instructions_text', 'gt_ingredients_text', 'gt_instructions_text']] = df.apply(
        lambda row: extract_ingredients_and_instructions(row[llm_final_answer], row[gt_final_answer], row['llm_response_sources'],
                                                         row['gt_sources_extracted']), axis=1, result_type="expand")

In [ ]:
# Converting each list into a single string joined by a newline character and store it in a new column

df['gt_ingredients_text_split'] = df['gt_ingredients_text'].apply(
lambda item: "\n".join(item) if isinstance(item, list) else "")

df['gt_instructions_text_split'] = df['gt_instructions_text'].apply(
lambda item: "\n".join(item) if isinstance(item, list) else "")

df['llm_ingredients_text_split'] = df['llm_ingredients_text'].apply(
    lambda item: "\n".join(item) if isinstance(item, list) else "")

df['llm_instructions_text_split'] = df['llm_instructions_text'].apply(
    lambda item: "\n".join(item) if isinstance(item, list) else "")

In [ ]:
# Function to produce Ingredients & Instructions Score

def evaluate_content(llm_text, gt_text, casenum, threshold_in):
    """
    Evaluates the semantic similarity between sentences in LLM-generated text and
    ground truth text to produce a content score.

    The function splits both texts into sentences, encodes them using a SentenceTransformer model,
    and calculates cosine similarity between all pairs of LLM and ground truth sentences.
    It then matches sentences based on a provided similarity threshold and calculates a score
    that rewards matched sentences and penalizes extra LLM-generated sentences and
    missing ground truth sentences. Transition words (defined in `ignore_words`) are
    excluded from penalty calculations.

    Args:
        llm_text (str): The text generated by the Language Model.
        gt_text (str): The ground truth text.
        casenum (str): A unique identifier for the case being evaluated. This is
            included in the output DataFrame.
        threshold_in (float): The cosine similarity threshold (between 0 and 1)
            above which an LLM sentence is considered a match for a ground truth sentence.

    Returns:
        pandas.DataFrame: A DataFrame with a single row containing the evaluation metrics:
            - 'case_id': The provided `casenum`.
            - 'sentences_score': The overall content score (between 0 and 2).
            - 'num_gt_sentences': The total number of sentences in the ground truth text.
            - 'points_per_sentence': The base points awarded per matched ground truth sentence.
            - 'num_llm_sentences': The total number of sentences in the LLM-generated text.
            - 'correct_sentences_score': The total score from correctly matched sentences.
            - 'extra_sentences_penalty': The penalty applied for extra sentences in the LLM text.
            - 'gt_not_in_llm_penalty': The penalty applied for ground truth sentences not found in the LLM text.
            - 'num_equal_sentences': The number of exactly matching sentences (case-sensitive).
            - 'matched_sentences_count': The number of ground truth sentences with a similarity above the threshold in the LLM text.
            - 'gt_not_in_llm_count': The number of ground truth sentences with no similar counterpart in the LLM text (above the threshold, excluding transition sentences).
            - 'extra_sentences_count': The number of LLM sentences with no similar counterpart in the ground truth text (above the threshold, excluding transition sentences).
            - 'gt_transition_sentence_count': The number of ground truth sentences identified as transition sentences (based on `ignore_words`).
            - 'llm_transition_sentence_count': The number of LLM sentences identified as transition sentences.
            - 'gt_sentences': The ground truth text with sentences separated by newlines.
            - 'llm_sentences': The LLM-generated text with sentences separated by newlines.
            - 'matched_sentences': Pairs of matched ground truth and LLM sentences, each pair separated by a newline.
            - 'extra_sentences': LLM sentences that did not meet the similarity threshold with any ground truth sentence.
            - 'gt_not_in_llm_sentences': Ground truth sentences that did not meet the similarity threshold with any LLM sentence.

    Raises:
        Exception: If any error occurs during the evaluation process, the error is printed,
            and a DataFrame with default zero/empty values is returned.
    """

    model = SentenceTransformer('all-mpnet-base-v2')
    threshold = threshold_in

    try:
        if len(llm_text) > 0 and len(gt_text) > 0 and llm_text != "None" and gt_text != "None":
            # Split ground truth and LLM text into sentences
            gt_sentences = gt_text.split("\n")
            gt_sentences = [sent.strip() for sent in gt_sentences if sent.strip()]

            llm_sentences = llm_text.split("\n")
            llm_sentences = [sent.strip() for sent in llm_sentences if sent.strip()]

            # Encode LLM and ground truth sentences
            llm_embeddings = model.encode(llm_sentences)
            gt_embeddings = model.encode(gt_sentences)

            # Calculate similarity for each LLM sentence to all ground truth sentences
            similarities = util.cos_sim(llm_embeddings, gt_embeddings)

            # Initialize lists to store matched and extra sentences
            matched_sentences = []
            extra_sentences = []
            gt_not_in_llm_sentences = []

            # Count sentences with high similarity and penalize for extra sentences
            matched_sentences_count = 0
            extra_sentences_count = 0
            gt_not_in_llm_count = 0
            gt_transition_sentence_count = 0
            llm_transition_sentence_count = 0

            # To ignore transition sentences on penalty calculation
            ignore_words = [",", "\""]
            ignore_pattern = re.compile(r"|".join(ignore_words), re.IGNORECASE)  # Case-insensitive pattern

            for gt_index, gt_sentence in enumerate(gt_sentences):
                max_sim_index = np.argmax(similarities[:, gt_index])
                if similarities[max_sim_index, gt_index] > threshold:
                    matched_sentences_count += 1
                    matched_pair = (
                        f"{gt_sentence}\n",
                        f"{llm_sentences[max_sim_index]}\n"
                    )
                    matched_sentences.append(matched_pair)
                else:
                    if ignore_pattern.search(gt_sentence):
                        gt_transition_sentence_count += 1
                    else:
                        gt_not_in_llm_count += 1
                        gt_not_in_llm_sentences.append(gt_sentence + "\n")

            for i, llm_similarities in enumerate(similarities):
                if max(llm_similarities) < threshold:
                    if ignore_pattern.search(llm_sentences[i]):
                        llm_transition_sentence_count += 1
                    else:
                        extra_sentences_count += 1
                        extra_sentences.append(llm_sentences[i] + "\n")


            if (len(gt_sentences) - gt_transition_sentence_count) == 0:
                points_per_sentence = 0.00
            else:
                points_per_sentence = round(2 / (len(gt_sentences) - gt_transition_sentence_count), 4)

            # Calculate the score
            correct_sentences_score = matched_sentences_count * points_per_sentence
            extra_sentences_penalty = extra_sentences_count * 0.10 * points_per_sentence # Apply penalty for extra sentences
            gt_not_in_llm_penalty = gt_not_in_llm_count * 0.20 * points_per_sentence # Apply penalty for sentences in GT not in LLM

            # Store number of sentences
            num_gt_sentences = len(gt_sentences)
            num_llm_sentences = len(llm_sentences)

            # Count the number of exactly matching sentences
            num_equal_sentences = sum(1 for gt, llm in matched_sentences if gt.strip() == llm.strip())

            score = round(max(0, min(2, correct_sentences_score - extra_sentences_penalty - gt_not_in_llm_penalty)), 3)

        else:
            points_per_sentence = 0
            score = 0
            gt_sentences = []
            llm_sentences = []
            correct_sentences_score = 0
            extra_sentences_penalty = 0
            num_gt_sentences = 0
            num_llm_sentences = 0
            matched_sentences = []
            extra_sentences = []
            num_equal_sentences = 0
            gt_not_in_llm_sentences = []
            gt_not_in_llm_count = 0
            gt_not_in_llm_penalty = 0
            matched_sentences_count = 0
            extra_sentences_count = 0
            gt_transition_sentence_count = 0
            llm_transition_sentence_count = 0

    except Exception as e:  # Broad exception handling to catch any errors
        print(f"Error in case {casenum}: {e}")  # Log the error for debugging

        # Return empty strings/zeros for all columns
        data = {
            'Case Number': casenum,
            'sentences_score': 0,
            'num_gt_sentences': 0,
            'points_per_sentence': 0,
            'num_llm_sentences': 0,
            'correct_sentences_score': 0,
            'extra_sentences_penalty': 0,
            'gt_not_in_llm_penalty': 0,
            'num_equal_sentences': 0,
            'matched_sentences_count': 0,
            'gt_not_in_llm_count': 0,
            'extra_sentences_count': 0,
            'gt_transition_sentence_count': 0,
            'llm_transition_sentence_count': 0,
            'gt_sentences': "",
            'llm_sentences': "",
            'matched_sentences': "",
            'extra_sentences': "",
            'gt_not_in_llm_sentences': ""
        }
        return pd.DataFrame(data, index=[0])

    data = {
        # MAKE SURE YOU'RE GIVING 'casenum' THE SAME NAME AS in YOUR 'df' AS YOU WILL BE MERGING LATER
        'case_id': casenum,
        'sentences_score': max(0, min(2, score)),
        'num_gt_sentences': num_gt_sentences,
        'points_per_sentence': points_per_sentence,
        'num_llm_sentences': num_llm_sentences,
        'correct_sentences_score': correct_sentences_score,
        'extra_sentences_penalty': extra_sentences_penalty,
        'gt_not_in_llm_penalty': gt_not_in_llm_penalty,
        'num_equal_sentences': num_equal_sentences,
        'matched_sentences_count': matched_sentences_count,
        'gt_not_in_llm_count': gt_not_in_llm_count,
        'extra_sentences_count': extra_sentences_count,
        'gt_transition_sentence_count' : gt_transition_sentence_count,
        'llm_transition_sentence_count' : llm_transition_sentence_count,

        # Join list elements into a single string
        'gt_sentences': "\n".join(gt_sentences),
        'llm_sentences': "\n".join(llm_sentences),
        'matched_sentences': "\n".join([f"{gt}\n{llm}" for gt, llm in matched_sentences]),
        'extra_sentences': "\n".join(extra_sentences),
        'gt_not_in_llm_sentences': "\n".join(gt_not_in_llm_sentences)
    }

    return pd.DataFrame(data, index=[0])

In [ ]:
# Semantic Similarity Threshold default value is 85%

results_ingredients = df.apply(
lambda row: evaluate_content(row['llm_ingredients_text_split'], row['gt_ingredients_text_split'], row['case_id'], 0.85), axis=1)

In [ ]:
# Semantic Similarity Threshold default value is 75%

results_instructions = df.apply(
lambda row: evaluate_content(row['llm_instructions_text_split'], row['gt_instructions_text_split'], row['case_id'], 0.75), axis=1)

In [ ]:
df_ingredients = pd.concat(results_ingredients.tolist(), ignore_index=True)
df_instructions = pd.concat(results_instructions.tolist(), ignore_index=True)

In [ ]:
df = df.merge(df_ingredients, on='case_id', how='left')

In [ ]:
# Identify potential overlapping columns (excluding case_id)
overlapping_cols_ingredients = list(set(df.columns) & set(df_ingredients.columns) - {'case_id'})
overlapping_cols_instructions = list(set(df.columns) & set(df_instructions.columns) - {'case_id'})

# Rename overlapping columns in df_ingredients
df_ingredients = df_ingredients.rename(columns={col: f"{col}_ingredient" for col in overlapping_cols_ingredients})

# Rename overlapping columns in df_instructions
df_instructions = df_instructions.rename(columns={col: f"{col}_instruction" for col in overlapping_cols_instructions})

# Perform the merges
final_df = df.merge(df_ingredients, on='case_id', how='left')
final_df = final_df.merge(df_instructions, on='case_id', how='left')

In [ ]:
# Dropping duplicate columns

columns_to_drop = ['sentences_score', 'num_gt_sentences', 'points_per_sentence', 'num_llm_sentences', 'correct_sentences_score', 'extra_sentences_penalty', 'gt_not_in_llm_penalty', 'num_equal_sentences', 'matched_sentences_count', 'gt_not_in_llm_count', 'extra_sentences_count', 'gt_transition_sentence_count', 'llm_transition_sentence_count', 'gt_sentences', 'llm_sentences', 'matched_sentences', 'extra_sentences', 'gt_not_in_llm_sentences']

final_df.drop(columns=columns_to_drop, axis=1, inplace=True)

In [ ]:
final_df['final_score'] = final_df['source_score'] + final_df['sentences_score_ingredient'] + final_df['sentences_score_instruction'] + final_df['rephraser_semantic_similarity']

#### Edge case for "No Answer Available"

In [ ]:
def no_answer_case(model, df, gt_col, llm_col, similarity_threshold=0.2):
    """
    Identifies and scores cases where both the ground truth and the LLM indicate
    an inability to answer a question.

    This function iterates through a DataFrame, comparing the answers in a ground
    truth column and an LLM output column. It checks if both answers express
    a lack of information using either semantic similarity to the phrase
    "No answer to this question" (above a specified threshold) or by containing
    predefined "no answer" keywords. If both conditions are met for a row,
    specific score columns ('source_score', 'sentences_score_ingredient',
    'sentences_score_instruction', 'final_score') in that row are set to predefined
    positive values, indicating a correctly identified "no answer" scenario.

    Args:
        df (pd.DataFrame): The DataFrame containing the ground truth and LLM answers.
        gt_col (str): The name of the column containing the ground truth answers.
        llm_col (str): The name of the column containing the LLM-generated answers.
        similarity_threshold (float, optional): The minimum cosine similarity
            (between 0 and 1) to the phrase "No answer to this question" for an
            answer to be considered a "no answer" case based on semantic similarity.

    Returns:
        pd.DataFrame: The input DataFrame with updated score values for rows where
                      both ground truth and LLM indicate an inability to answer.
                      Specific score columns are set to 1, 2, 2, and 5 respectively
                      for these "no answer" cases.

    Raises:
        KeyError: If the specified `gt_col` or `llm_col` are not found in the DataFrame.
        Exception: For any other unexpected error during row processing, an error
                   message is printed, and the row is skipped.
    """


    target_phrase = "No answer to this question"
    no_answer_keywords = ["no answer", "cannot answer", "not able to answer",
                           "not able to", "not found", "not available",
                           "no relevant information", "does not contain",
                           "unable to find"]

    for index, row in df.iterrows():
        try:
            gt_answer = str(row[gt_col]).strip().lower()
            llm_answer = str(row[llm_col]).strip().lower()

            gt_similarity = calculate_similarity(model, gt_answer, target_phrase)
            llm_similarity = calculate_similarity(model, llm_answer, target_phrase)

            gt_has_keyword = any(keyword in gt_answer for keyword in no_answer_keywords)
            llm_has_keyword = any(keyword in llm_answer for keyword in no_answer_keywords)

            # print(f"Index: {index}, GT Similarity: {gt_similarity:.4f}, LLM Similarity: {llm_similarity:.4f}, GT Keywords: {gt_has_keyword}, LLM Keywords: {llm_has_keyword}")

            if (gt_similarity >= similarity_threshold and llm_similarity >= similarity_threshold) or (gt_has_keyword and llm_has_keyword):
                df.loc[index, 'source_score'] = 1
                df.loc[index, 'sentences_score_ingredient'] = 2
                df.loc[index, 'sentences_score_instruction'] = 2
                df.loc[index, 'final_score'] = 5
        except KeyError as e:
            print(f"Error: Column '{e}' not found in the DataFrame for row {index}. Skipping row.")
        except Exception as e:
            print(f"An unexpected error occurred while processing row {index}: {e}")
    return df

### Final Result

In [ ]:
no_answer_case(model,
        df=final_df,
        gt_col="ground_truth_final_answer",
        llm_col="gemini_final_answer_llm",
        similarity_threshold=0.5  # You can adjust the threshold if needed
    )

In [ ]:
final_df.to_csv('chef-advisor-llm-results.csv', sep=sep, index=False)

## Vertex GenAI Evaluation Service

### Import libraries

In [ ]:
from vertexai.evaluation import EvalTask, PointwiseMetric, PointwiseMetricPromptTemplate
from vertexai.preview.evaluation import notebook_utils

### Rephraser Evaluation

#### Define Evaluation Metric(s) Based on Your Use Case

In [ ]:
eval_dataset_rephraser = pd.DataFrame(
    {
        "context": final_df["query"],
        "reference": final_df[gt_rephrased_query],
        "response": final_df[llm_rephrased_query],
    }
)
     

In [ ]:
# Customized evaluation criteria for Rephraser
search_optimized_rephrasing = PointwiseMetric(
    metric="search_optimized_rephrasing",
    metric_prompt_template= PointwiseMetricPromptTemplate(
        input_variables=["context", "reference", "response"],
        criteria={
            "conciseness": "The rephrased query is 20 words or less.",
            "core_ingredients_tastes": "The rephrased query accurately captures the core ingredients mentioned in the original query.",
            "relevant_tastes": "The rephrased query accurately captures the key tastes or flavor profiles mentioned in the original query (e.g., sweet, sour, spicy).",
            "search_relevance": "The rephrased query consists of keywords that are highly likely to retrieve relevant documents in a search engine for a chef seeking recipes.",
            "no_extraneous_info": "The rephrased query does not include any information beyond the core ingredients and tastes.",
        },
        rating_rubric={
            "1": "The rephrased query perfectly meets all criteria: it is 20 words or less, accurately captures all core ingredients and relevant tastes, consists of highly \
                  relevant search keywords with no extraneous information.",
            "0.75": "The rephrased query mostly meets the criteria. It is likely 20 words or less and captures most of the core ingredients and relevant tastes as effective \
                    search keywords, with minimal or no extraneous information.",
            "0.5": "The rephrased query partially meets the criteria. It might exceed 20 words or miss some core ingredients or relevant tastes. The search keywords might be \
                    somewhat relevant but could be improved.",
            "0.25": "The rephrased query poorly meets the criteria. It likely exceeds 20 words, misses significant core ingredients or relevant tastes, and the search keywords \
                     are not very effective for retrieval.",
            "0": "The rephrased query fails to meet most or all of the criteria. It is significantly longer than 20 words, misses crucial core ingredients and relevant tastes, \
                  provides irrelevant or ineffective search keywords, or includes significant extraneous information.",
        },
    ),
)

In [ ]:
eval_result = EvalTask(
    dataset=eval_dataset_rephraser, metrics=[search_optimized_rephrasing], experiment="genai-eval-service-rephraser-1"
).evaluate()

In [ ]:
notebook_utils.display_eval_result(eval_result)

In [ ]:
rephraser_genai_eval_service_summary_metrics_df = pd.DataFrame(eval_result.summary_metrics, index=[0]) 
rephraser_genai_eval_service_metrics_table_df = eval_result.metrics_table

In [ ]:
rephraser_genai_eval_service_metrics_table_df.to_csv('chef-advisor-genai-eval-service-results-rephraser.csv', sep=sep, index=False)

### Clean Up

Delete ExperimentRun created by the evaluation.

In [ ]:
from google.cloud import aiplatform

aiplatform.ExperimentRun(
    run_name=eval_result.metadata["experiment_run"],
    experiment=eval_result.metadata["experiment"],
).delete()